In [ ]:
#Import libraries to open data file
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import sklearn
import xgboost
from xgboost import XGBClassifier
import matplotlib
import seaborn as sns
import itertools
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer, balanced_accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.feature_selection import RFE
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from statistics import mean as stat_mean
from numpy import mean
from numpy import std

### For AFFF-GW

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
#print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
# Split into test and training, and evaluate optimal hyperparameters for Random Forest
totalR = 10
rs = np.arange(1, totalR + 1)

#Define hyperparameters
gamma = [0,0.1,0.2,0.4,0.8,1]
learning_rate = [0.01, 0.1, 0.2, 0.5]
max_depth = [5,6,7]
n_estimators = [50,100,150]
reg_alpha = [0,0.5, 10, 200]
reg_lambda = [0,0.5, 10, 200]

grid = dict(gamma=gamma, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, reg_alpha=reg_alpha, reg_lambda=reg_lambda)

balanced_accuracy_scorer = make_scorer(balanced_accuracy_score)
# Initialize arrays to store results
test_balanced_accuracy = np.empty(len(rs))
train_balanced_accuracy = np.empty(len(rs))



for j in rs:
    X_train, X_test, y_train, y_test = train_test_split(
        data_1, target_1, test_size=0.2, stratify=target_1, random_state=j)
    
    # Create xgboost
    xgbo = XGBClassifier()
    
    # Create repeated stratified k-fold cross-validator
    cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=5, random_state=42)
    
    # Perform hyperparameter tuning with GridSearchCV
    grid_search = GridSearchCV(estimator=xgbo, param_grid=grid, n_jobs=-1, cv=cv, scoring=balanced_accuracy_scorer, error_score=0)
    grid_result = grid_search.fit(X_train, y_train.ravel())
    
    print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

    # Get the best Logsistic regression model from GridSearchCV
    best_rf = grid_search.best_estimator_

    # Compute balanced accuracy on the training and test data
    train_predictions = best_rf.predict(X_train)
    test_predictions = best_rf.predict(X_test)
    
    train_ba = balanced_accuracy_score(y_train, train_predictions)
    test_ba = balanced_accuracy_score(y_test, test_predictions)
    
    train_balanced_accuracy[j-1] = train_ba
    test_balanced_accuracy[j-1] = test_ba

In [ ]:
# Plot balanced accuracy for different random states
plt.plot(rs, test_balanced_accuracy, label='Testing dataset Balanced Accuracy')
plt.plot(rs, train_balanced_accuracy, label='Training dataset Balanced Accuracy')

plt.legend(fontsize=12)
plt.title('XGB Classifier Balanced Accuracy', fontsize=14)
plt.xlabel('Random State', fontsize=14)
plt.ylabel('Balanced Accuracy', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.savefig('XGB-Tuning-NORM-GW.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the value of best balanced accuracy and the corresponding random state
best_test_balanced_accuracy = np.max(test_balanced_accuracy)
best_random_state = rs[np.argmax(test_balanced_accuracy)]
print("Best Testing Balanced Accuracy:", best_test_balanced_accuracy)
print("Corresponding Random State:", best_random_state)

In [ ]:
print(train_balanced_accuracy)
print(test_balanced_accuracy)

In [ ]:
#Now running optimized model over the entire dataset
model = XGBClassifier(gamma = 0.8, learning_rate = 0.5, max_depth = 5, n_estimators = 50, reg_alpha = 0, reg_lambda = 0)
# fit the model
model.fit(data_1, target_1.ravel())

# get importance
importance = model.feature_importances_
# summarize feature importance
for i,v in enumerate(importance):
    print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
plt.bar([x for x in range(len(importance))], importance)
plt.show()

In [ ]:
# Initialize variables to store feature importances across iterations
all_importances = []

# Number of iterations
num_iterations = 100

for iteration in range(num_iterations):
    # Create and fit the model
    model = XGBClassifier(gamma = 0.8, learning_rate = 0.5, max_depth = 5, n_estimators = 50, reg_alpha = 0, reg_lambda = 0)
    model.fit(data_1, target_1.ravel())

    # Get importance and store in the list
    importance = model.feature_importances_
    all_importances.append(importance)

# Convert the list to a NumPy array for easy calculations
all_importances = np.array(all_importances)

# Calculate average and standard deviation of feature importances
average_importance = np.mean(all_importances, axis=0)
std_importance = np.std(all_importances, axis=0)

# Summarize and print feature importance statistics
for i, (avg_score, std_score) in enumerate(zip(average_importance, std_importance)):
    print(f'Feature {i}: Average Score: {avg_score:.5f}, Standard Deviation: {std_score:.5f}')

# Plot the average feature importance
plt.bar([x for x in range(len(average_importance))], average_importance, yerr=std_importance, capsize=5)
plt.xlabel('Feature Index')
plt.ylabel('Average Feature Importance')
plt.title('Average Feature Importance with Standard Deviation')
plt.show()

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

feature_importance_pairs = [(avg, std, name) for avg, std, name in zip(average_importance, std_importance, feature_names)]

# Sort the list by average importance in descending order
feature_importance_pairs.sort(reverse=True, key=lambda x: x[0])

# Print the output for the top 10 features
print("Top 10 Features:")
for avg, std, name in feature_importance_pairs[:10]:
    print(f'Feature: {name}, Mean Score: {avg:.5f}, Standard Deviation: {std:.5f}')

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Get indices of the top 10 features based on mean importance
top_10_indices = np.argsort(average_importance)[::-1][:10]

# Print top 10 features along with their average and standard deviation
print("Top 10 Features:")
for i in top_10_indices:
    print(f'Feature: {feature_names[i]}, Mean Score: {average_importance[i]:.5f}, Standard Deviation: {std_importance[i]:.5f}')

In [ ]:
# Rank features by importance
sorted_feature_importance_pairs = sorted(feature_importance_pairs, reverse=True, key=lambda x: x[0])

# Extract feature names and importances
sorted_importances, _, sorted_feature_names = zip(*sorted_feature_importance_pairs)

# Convert feature importances to a numeric data type
sorted_importances = np.array(sorted_importances, dtype=float)

# Calculate the difference between subsequent feature importances
importance_diff = np.diff(sorted_importances)

# Find the index where the largest decrease occurs
largest_decrease_index = np.argmax(importance_diff)


# Store the top features and scores in a DataFrame
df_top_features = pd.DataFrame({'Feature': [], 'Score': []})

# Print the top features until the largest decrease
print("Top Features with Significant Importance:")
for i, (avg, std, name) in enumerate(sorted_feature_importance_pairs[:largest_decrease_index + 1], 1):
    try:
        avg_float = float(avg)
        print(f"{i}. Feature: {name}, Mean Score: {avg_float:.5f}, Standard Deviation: {std:.5f}")
        df_top_features = df_top_features.append({'Feature': name, 'Score': avg_float}, ignore_index=True)
    except ValueError:
        print(f"{i}. Feature: {name}, Mean Score: {avg}, Standard Deviation: {std} (non-numeric)")

# Save the df_top_features DataFrame to an Excel file
df_top_features.to_excel('top_features_scores_norm_GW.xlsx', index=False)

# Plot feature importance
plt.figure(figsize=(200, 60))
plt.bar(range(len(sorted_importances)), sorted_importances, tick_label=sorted_feature_names)
plt.xticks(rotation=90)
plt.title("Feature Importances")
# Save the figure as a PNG file
plt.savefig('feature_importances_plot_ORDERED_norm_GW.png', bbox_inches='tight')

# Show the plot (optional)
plt.show()

In [ ]:
class_names=np.array([0.0,1.0])

#Generating confusion matrices (for entire dataset; normalized BAs)
def evaluate_model(data_x, data_y):
    k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

    predicted_targets = np.array([])
    actual_targets = np.array([])

    for train_ix, test_ix in k_fold.split(data_x, data_y):
        train_x, train_y, test_x, test_y = data_x[train_ix], data_y[train_ix], data_x[test_ix], data_y[test_ix]

        # Fit the classifier
        model_cv = XGBClassifier(gamma = 0.8, learning_rate = 0.5, max_depth = 5, n_estimators = 50, reg_alpha = 0, reg_lambda = 0)
        classifier = model_cv.fit(train_x, train_y.ravel())

        # Predict the labels of the test set samples
        predicted_labels = classifier.predict(test_x)

        predicted_targets = np.append(predicted_targets, predicted_labels)
        actual_targets = np.append(actual_targets, test_y)

    return predicted_targets, actual_targets

In [ ]:
def plot_confusion_matrix(predicted_labels_list, y_test_list):
    cnf_matrix = confusion_matrix(y_test_list, predicted_labels_list)
    np.set_printoptions(precision=2)

    # Plot non-normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, title='Confusion matrix, without normalization')
    plt.show()

    # Plot normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, normalize=True, title='Normalized confusion matrix')
    plt.show()

In [ ]:
def generate_confusion_matrix(cnf_matrix, classes, normalize=False, title='Confusion matrix'):
    if normalize:
        cnf_matrix = cnf_matrix.astype('float') / cnf_matrix.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.imshow(cnf_matrix, interpolation='nearest', cmap=plt.get_cmap('Blues'))
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cnf_matrix.max() / 2.
    
    for i, j in itertools.product(range(cnf_matrix.shape[0]), range(cnf_matrix.shape[1])):
        plt.text(j, i, format(cnf_matrix[i, j], fmt), horizontalalignment="center",
                 color="white" if cnf_matrix[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

    return cnf_matrix

In [ ]:
predicted_target, actual_target = evaluate_model(data_1, target_1)
plot_confusion_matrix(predicted_target, actual_target)

### For LL

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
#print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#Running randomized CV to narrow grid for GridSearch CV
#Modified from here: https://towardsdatascience.com/hyperparameter-tuning-the-random-forest-in-python-using-scikit-learn-28d2aa77dd74
random_grid = {'gamma': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4, 200],
              'learning_rate': [0.01, 0.03, 0.06, 0.1, 0.15, 0.2, 0.25, 0.300000012, 0.4, 0.5, 0.6, 0.7],
              'max_depth': [5,6,7,8,9,10,11,12,13,14],
              'n_estimators': [50,65,80,100,115,130,150],
              'reg_alpha': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200],
              'reg_lambda': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200]}

print(random_grid)

In [ ]:
# Use the random grid to search for best hyperparameters
# First create the base model to tune
model_rnd = XGBClassifier()
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = model_rnd, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
rf_random.fit(data_1, target_1)
print(rf_random.best_params_)

In [ ]:
# Split into test and training, and evaluate optimal hyperparameters for Random Forest
totalR = 10
rs = np.arange(1, totalR + 1)

#Define hyperparameters
gamma = [0, 0.5, 0.6]
learning_rate = [0.1, 0.5, 0.6]
max_depth = [5, 8, 10]
n_estimators = [50, 60, 70]
reg_lambda = [0.5, 1.6, 2]
reg_alpha = [0.5, 0.8, 12.8]

grid = dict(gamma=gamma, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, reg_alpha=reg_alpha, reg_lambda=reg_lambda)

balanced_accuracy_scorer = make_scorer(balanced_accuracy_score)
# Initialize arrays to store results
test_balanced_accuracy = np.empty(len(rs))
train_balanced_accuracy = np.empty(len(rs))



for j in rs:
    X_train, X_test, y_train, y_test = train_test_split(
        data_1, target_1, test_size=0.2, stratify=target_1, random_state=j)
    
    # Create xgboost
    xgbo = XGBClassifier()
    
    # Create repeated stratified k-fold cross-validator
    cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=5, random_state=42)
    
    # Perform hyperparameter tuning with GridSearchCV
    grid_search = GridSearchCV(estimator=xgbo, param_grid=grid, n_jobs=-1, cv=cv, scoring=balanced_accuracy_scorer, error_score=0)
    grid_result = grid_search.fit(X_train, y_train.ravel())
    
    print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

    # Get the best Logsistic regression model from GridSearchCV
    best_rf = grid_search.best_estimator_

    # Compute balanced accuracy on the training and test data
    train_predictions = best_rf.predict(X_train)
    test_predictions = best_rf.predict(X_test)
    
    train_ba = balanced_accuracy_score(y_train, train_predictions)
    test_ba = balanced_accuracy_score(y_test, test_predictions)
    
    train_balanced_accuracy[j-1] = train_ba
    test_balanced_accuracy[j-1] = test_ba

In [ ]:
# Plot balanced accuracy for different random states
plt.plot(rs, test_balanced_accuracy, label='Testing dataset Balanced Accuracy')
plt.plot(rs, train_balanced_accuracy, label='Training dataset Balanced Accuracy')

plt.legend(fontsize=12)
plt.title('XGB Classifier Balanced Accuracy', fontsize=14)
plt.xlabel('Random State', fontsize=14)
plt.ylabel('Balanced Accuracy', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.savefig('XGB-Tuning-NORM-LL.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the value of best balanced accuracy and the corresponding random state
best_test_balanced_accuracy = np.max(test_balanced_accuracy)
best_random_state = rs[np.argmax(test_balanced_accuracy)]
print("Best Testing Balanced Accuracy:", best_test_balanced_accuracy)
print("Corresponding Random State:", best_random_state)

In [ ]:
print(train_balanced_accuracy)
print(test_balanced_accuracy)

In [ ]:
#Now running optimized model over the entire dataset
model = XGBClassifier(gamma = 0, learning_rate = 0.5, max_depth = 5, n_estimators = 50, reg_alpha = 0.5, reg_lambda = 1.6)
# fit the model
model.fit(data_1, target_1.ravel())

# get importance
importance = model.feature_importances_
# summarize feature importance
for i,v in enumerate(importance):
    print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
plt.bar([x for x in range(len(importance))], importance)
plt.show()

In [ ]:
# Initialize variables to store feature importances across iterations
all_importances = []

# Number of iterations
num_iterations = 100

for iteration in range(num_iterations):
    # Create and fit the model
    model = XGBClassifier(gamma = 0, learning_rate = 0.5, max_depth = 5, n_estimators = 50, reg_alpha = 0.5, reg_lambda = 1.6)
    model.fit(data_1, target_1.ravel())

    # Get importance and store in the list
    importance = model.feature_importances_
    all_importances.append(importance)

# Convert the list to a NumPy array for easy calculations
all_importances = np.array(all_importances)

# Calculate average and standard deviation of feature importances
average_importance = np.mean(all_importances, axis=0)
std_importance = np.std(all_importances, axis=0)

# Summarize and print feature importance statistics
for i, (avg_score, std_score) in enumerate(zip(average_importance, std_importance)):
    print(f'Feature {i}: Average Score: {avg_score:.5f}, Standard Deviation: {std_score:.5f}')

# Plot the average feature importance
plt.bar([x for x in range(len(average_importance))], average_importance, yerr=std_importance, capsize=5)
plt.xlabel('Feature Index')
plt.ylabel('Average Feature Importance')
plt.title('Average Feature Importance with Standard Deviation')
plt.show()

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

feature_importance_pairs = [(avg, std, name) for avg, std, name in zip(average_importance, std_importance, feature_names)]

# Sort the list by average importance in descending order
feature_importance_pairs.sort(reverse=True, key=lambda x: x[0])

# Print the output for the top 10 features
print("Top 10 Features:")
for avg, std, name in feature_importance_pairs[:10]:
    print(f'Feature: {name}, Mean Score: {avg:.5f}, Standard Deviation: {std:.5f}')

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Get indices of the top 10 features based on mean importance
top_10_indices = np.argsort(average_importance)[::-1][:10]

# Print top 10 features along with their average and standard deviation
print("Top 10 Features:")
for i in top_10_indices:
    print(f'Feature: {feature_names[i]}, Mean Score: {average_importance[i]:.5f}, Standard Deviation: {std_importance[i]:.5f}')

In [ ]:
# Rank features by importance
sorted_feature_importance_pairs = sorted(feature_importance_pairs, reverse=True, key=lambda x: x[0])

# Extract feature names and importances
sorted_importances, _, sorted_feature_names = zip(*sorted_feature_importance_pairs)

# Convert feature importances to a numeric data type
sorted_importances = np.array(sorted_importances, dtype=float)

# Calculate the difference between subsequent feature importances
importance_diff = np.diff(sorted_importances)

# Find the index where the largest decrease occurs
largest_decrease_index = np.argmax(importance_diff)


# Store the top features and scores in a DataFrame
df_top_features = pd.DataFrame({'Feature': [], 'Score': []})

# Print the top features until the largest decrease
print("Top Features with Significant Importance:")
for i, (avg, std, name) in enumerate(sorted_feature_importance_pairs[:largest_decrease_index + 1], 1):
    try:
        avg_float = float(avg)
        print(f"{i}. Feature: {name}, Mean Score: {avg_float:.5f}, Standard Deviation: {std:.5f}")
        df_top_features = df_top_features.append({'Feature': name, 'Score': avg_float}, ignore_index=True)
    except ValueError:
        print(f"{i}. Feature: {name}, Mean Score: {avg}, Standard Deviation: {std} (non-numeric)")

# Save the df_top_features DataFrame to an Excel file
df_top_features.to_excel('top_features_scores_norm_ll.xlsx', index=False)

# Plot feature importance
plt.figure(figsize=(200, 60))
plt.bar(range(len(sorted_importances)), sorted_importances, tick_label=sorted_feature_names)
plt.xticks(rotation=90)
plt.title("Feature Importances")
# Save the figure as a PNG file
plt.savefig('feature_importances_plot_ORDERED_norm_LL.png', bbox_inches='tight')

# Show the plot (optional)
plt.show()

In [ ]:
class_names=np.array([0.0,1.0])

#Generating confusion matrices (for entire dataset; normalized BAs)
def evaluate_model(data_x, data_y):
    k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

    predicted_targets = np.array([])
    actual_targets = np.array([])

    for train_ix, test_ix in k_fold.split(data_x, data_y):
        train_x, train_y, test_x, test_y = data_x[train_ix], data_y[train_ix], data_x[test_ix], data_y[test_ix]

        # Fit the classifier
        model_cv = XGBClassifier(gamma = 0, learning_rate = 0.5, max_depth = 5, n_estimators = 50, reg_alpha = 0.5, reg_lambda = 1.6)
        classifier = model_cv.fit(train_x, train_y.ravel())

        # Predict the labels of the test set samples
        predicted_labels = classifier.predict(test_x)

        predicted_targets = np.append(predicted_targets, predicted_labels)
        actual_targets = np.append(actual_targets, test_y)

    return predicted_targets, actual_targets

In [ ]:
def plot_confusion_matrix(predicted_labels_list, y_test_list):
    cnf_matrix = confusion_matrix(y_test_list, predicted_labels_list)
    np.set_printoptions(precision=2)

    # Plot non-normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, title='Confusion matrix, without normalization')
    plt.show()

    # Plot normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, normalize=True, title='Normalized confusion matrix')
    plt.show()

In [ ]:
def generate_confusion_matrix(cnf_matrix, classes, normalize=False, title='Confusion matrix'):
    if normalize:
        cnf_matrix = cnf_matrix.astype('float') / cnf_matrix.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.imshow(cnf_matrix, interpolation='nearest', cmap=plt.get_cmap('Blues'))
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cnf_matrix.max() / 2.
    
    for i, j in itertools.product(range(cnf_matrix.shape[0]), range(cnf_matrix.shape[1])):
        plt.text(j, i, format(cnf_matrix[i, j], fmt), horizontalalignment="center",
                 color="white" if cnf_matrix[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

    return cnf_matrix

In [ ]:
predicted_target, actual_target = evaluate_model(data_1, target_1)
plot_confusion_matrix(predicted_target, actual_target)

### BL

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
#print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#Running randomized CV to narrow grid for GridSearch CV
#Modified from here: https://towardsdatascience.com/hyperparameter-tuning-the-random-forest-in-python-using-scikit-learn-28d2aa77dd74
random_grid = {'gamma': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4, 200],
              'learning_rate': [0.01, 0.03, 0.06, 0.1, 0.15, 0.2, 0.25, 0.300000012, 0.4, 0.5, 0.6, 0.7],
              'max_depth': [5,6,7,8,9,10,11,12,13,14],
              'n_estimators': [50,65,80,100,115,130,150],
              'reg_alpha': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200],
              'reg_lambda': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200]}

print(random_grid)

In [ ]:
# Use the random grid to search for best hyperparameters
# First create the base model to tune
model_rnd = XGBClassifier()
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = model_rnd, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
rf_random.fit(data_1, target_1)
print(rf_random.best_params_)

In [ ]:
# Split into test and training, and evaluate optimal hyperparameters for Random Forest
totalR = 10
rs = np.arange(1, totalR + 1)

#Define hyperparameters
gamma = [0.1,0.5,25.6]
learning_rate = [0.1, 0.5, 0.8]
max_depth = [6, 8, 10]
n_estimators = [100, 115, 130]
reg_alpha = [0.1, 0.8, 1.2]
reg_lambda = [0.1, 0.2, 0.3]

grid = dict(gamma=gamma, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, reg_alpha=reg_alpha, reg_lambda=reg_lambda)

balanced_accuracy_scorer = make_scorer(balanced_accuracy_score)
# Initialize arrays to store results
test_balanced_accuracy = np.empty(len(rs))
train_balanced_accuracy = np.empty(len(rs))



for j in rs:
    X_train, X_test, y_train, y_test = train_test_split(
        data_1, target_1, test_size=0.2, stratify=target_1, random_state=j)
    
    # Create xgboost
    xgbo = XGBClassifier()
    
    # Create repeated stratified k-fold cross-validator
    cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=5, random_state=42)
    
    # Perform hyperparameter tuning with GridSearchCV
    grid_search = GridSearchCV(estimator=xgbo, param_grid=grid, n_jobs=-1, cv=cv, scoring=balanced_accuracy_scorer, error_score=0)
    grid_result = grid_search.fit(X_train, y_train.ravel())
    
    print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

    # Get the best Logsistic regression model from GridSearchCV
    best_rf = grid_search.best_estimator_

    # Compute balanced accuracy on the training and test data
    train_predictions = best_rf.predict(X_train)
    test_predictions = best_rf.predict(X_test)
    
    train_ba = balanced_accuracy_score(y_train, train_predictions)
    test_ba = balanced_accuracy_score(y_test, test_predictions)
    
    train_balanced_accuracy[j-1] = train_ba
    test_balanced_accuracy[j-1] = test_ba

In [ ]:
# Plot balanced accuracy for different random states
plt.plot(rs, test_balanced_accuracy, label='Testing dataset Balanced Accuracy')
plt.plot(rs, train_balanced_accuracy, label='Training dataset Balanced Accuracy')

plt.legend(fontsize=12)
plt.title('XGB Classifier Balanced Accuracy', fontsize=14)
plt.xlabel('Random State', fontsize=14)
plt.ylabel('Balanced Accuracy', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.savefig('XGB-Tuning-NORM-BL.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the value of best balanced accuracy and the corresponding random state
best_test_balanced_accuracy = np.max(test_balanced_accuracy)
best_random_state = rs[np.argmax(test_balanced_accuracy)]
print("Best Testing Balanced Accuracy:", best_test_balanced_accuracy)
print("Corresponding Random State:", best_random_state)

In [ ]:
print(train_balanced_accuracy)
print(test_balanced_accuracy)

In [ ]:
#Now running optimized model over the entire dataset
model = XGBClassifier(gamma = 0.1, learning_rate = 0.1, max_depth = 6, n_estimators = 100, reg_alpha = 0.1, reg_lambda = 0.2)
# fit the model
model.fit(data_1, target_1.ravel())

# get importance
importance = model.feature_importances_
# summarize feature importance
for i,v in enumerate(importance):
    print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
plt.bar([x for x in range(len(importance))], importance)
plt.show()

In [ ]:
# Initialize variables to store feature importances across iterations
all_importances = []

# Number of iterations
num_iterations = 100

for iteration in range(num_iterations):
    # Create and fit the model
    model = XGBClassifier(gamma = 0.1, learning_rate = 0.1, max_depth = 6, n_estimators = 100, reg_alpha = 0.1, reg_lambda = 0.2)
    model.fit(data_1, target_1.ravel())

    # Get importance and store in the list
    importance = model.feature_importances_
    all_importances.append(importance)

# Convert the list to a NumPy array for easy calculations
all_importances = np.array(all_importances)

# Calculate average and standard deviation of feature importances
average_importance = np.mean(all_importances, axis=0)
std_importance = np.std(all_importances, axis=0)

# Summarize and print feature importance statistics
for i, (avg_score, std_score) in enumerate(zip(average_importance, std_importance)):
    print(f'Feature {i}: Average Score: {avg_score:.5f}, Standard Deviation: {std_score:.5f}')

# Plot the average feature importance
plt.bar([x for x in range(len(average_importance))], average_importance, yerr=std_importance, capsize=5)
plt.xlabel('Feature Index')
plt.ylabel('Average Feature Importance')
plt.title('Average Feature Importance with Standard Deviation')
plt.show()

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

feature_importance_pairs = [(avg, std, name) for avg, std, name in zip(average_importance, std_importance, feature_names)]

# Sort the list by average importance in descending order
feature_importance_pairs.sort(reverse=True, key=lambda x: x[0])

# Print the output for the top 10 features
print("Top 10 Features:")
for avg, std, name in feature_importance_pairs[:10]:
    print(f'Feature: {name}, Mean Score: {avg:.5f}, Standard Deviation: {std:.5f}')

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Get indices of the top 10 features based on mean importance
top_10_indices = np.argsort(average_importance)[::-1][:10]

# Print top 10 features along with their average and standard deviation
print("Top 10 Features:")
for i in top_10_indices:
    print(f'Feature: {feature_names[i]}, Mean Score: {average_importance[i]:.5f}, Standard Deviation: {std_importance[i]:.5f}')

In [ ]:
# Rank features by importance
sorted_feature_importance_pairs = sorted(feature_importance_pairs, reverse=True, key=lambda x: x[0])

# Extract feature names and importances
sorted_importances, _, sorted_feature_names = zip(*sorted_feature_importance_pairs)

# Convert feature importances to a numeric data type
sorted_importances = np.array(sorted_importances, dtype=float)

# Calculate the difference between subsequent feature importances
importance_diff = np.diff(sorted_importances)

# Find the index where the largest decrease occurs
largest_decrease_index = np.argmax(importance_diff)


# Store the top features and scores in a DataFrame
df_top_features = pd.DataFrame({'Feature': [], 'Score': []})

# Print the top features until the largest decrease
print("Top Features with Significant Importance:")
for i, (avg, std, name) in enumerate(sorted_feature_importance_pairs[:largest_decrease_index + 1], 1):
    try:
        avg_float = float(avg)
        print(f"{i}. Feature: {name}, Mean Score: {avg_float:.5f}, Standard Deviation: {std:.5f}")
        df_top_features = df_top_features.append({'Feature': name, 'Score': avg_float}, ignore_index=True)
    except ValueError:
        print(f"{i}. Feature: {name}, Mean Score: {avg}, Standard Deviation: {std} (non-numeric)")

# Save the df_top_features DataFrame to an Excel file
df_top_features.to_excel('top_features_scores_norm_BL.xlsx', index=False)

# Plot feature importance
plt.figure(figsize=(200, 60))
plt.bar(range(len(sorted_importances)), sorted_importances, tick_label=sorted_feature_names)
plt.xticks(rotation=90)
plt.title("Feature Importances")
# Save the figure as a PNG file
plt.savefig('feature_importances_plot_ORDERED_norm_BL.png', bbox_inches='tight')

# Show the plot (optional)
plt.show()

In [ ]:
class_names=np.array([0.0,1.0])

#Generating confusion matrices (for entire dataset; normalized BAs)
def evaluate_model(data_x, data_y):
    k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

    predicted_targets = np.array([])
    actual_targets = np.array([])

    for train_ix, test_ix in k_fold.split(data_x, data_y):
        train_x, train_y, test_x, test_y = data_x[train_ix], data_y[train_ix], data_x[test_ix], data_y[test_ix]

        # Fit the classifier
        model_cv = XGBClassifier(gamma = 0.1, learning_rate = 0.1, max_depth = 6, n_estimators = 100, reg_alpha = 0.1, reg_lambda = 0.2)
        classifier = model_cv.fit(train_x, train_y.ravel())

        # Predict the labels of the test set samples
        predicted_labels = classifier.predict(test_x)

        predicted_targets = np.append(predicted_targets, predicted_labels)
        actual_targets = np.append(actual_targets, test_y)

    return predicted_targets, actual_targets

In [ ]:
def plot_confusion_matrix(predicted_labels_list, y_test_list):
    cnf_matrix = confusion_matrix(y_test_list, predicted_labels_list)
    np.set_printoptions(precision=2)

    # Plot non-normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, title='Confusion matrix, without normalization')
    plt.show()

    # Plot normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, normalize=True, title='Normalized confusion matrix')
    plt.show()

In [ ]:
def generate_confusion_matrix(cnf_matrix, classes, normalize=False, title='Confusion matrix'):
    if normalize:
        cnf_matrix = cnf_matrix.astype('float') / cnf_matrix.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.imshow(cnf_matrix, interpolation='nearest', cmap=plt.get_cmap('Blues'))
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cnf_matrix.max() / 2.
    
    for i, j in itertools.product(range(cnf_matrix.shape[0]), range(cnf_matrix.shape[1])):
        plt.text(j, i, format(cnf_matrix[i, j], fmt), horizontalalignment="center",
                 color="white" if cnf_matrix[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

    return cnf_matrix

In [ ]:
predicted_target, actual_target = evaluate_model(data_1, target_1)
plot_confusion_matrix(predicted_target, actual_target)

### WWTP

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
#print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
#print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#Running randomized CV to narrow grid for GridSearch CV
#Modified from here: https://towardsdatascience.com/hyperparameter-tuning-the-random-forest-in-python-using-scikit-learn-28d2aa77dd74
random_grid = {'gamma': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4, 200],
              'learning_rate': [0.01, 0.03, 0.06, 0.1, 0.15, 0.2, 0.25, 0.300000012, 0.4, 0.5, 0.6, 0.7],
              'max_depth': [5,6,7,8,9,10,11,12,13,14],
              'n_estimators': [50,65,80,100,115,130,150],
              'reg_alpha': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200],
              'reg_lambda': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200]}

print(random_grid)

In [ ]:
# Use the random grid to search for best hyperparameters
# First create the base model to tune
model_rnd = XGBClassifier()
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = model_rnd, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
rf_random.fit(data_1, target_1)
print(rf_random.best_params_)

In [ ]:
# Split into test and training, and evaluate optimal hyperparameters for Random Forest
totalR = 10
rs = np.arange(1, totalR + 1)

#Define hyperparameters
gamma = [0.6,0.8,1]
learning_rate = [0.1, 0.4, 0.8]
max_depth = [10, 12, 20]
n_estimators = [70,80,90]
reg_alpha = [0.5, 0.8, 10]
reg_lambda = [0.5, 0.8, 10]

grid = dict(gamma=gamma, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, reg_alpha=reg_alpha, reg_lambda=reg_lambda)

balanced_accuracy_scorer = make_scorer(balanced_accuracy_score)
# Initialize arrays to store results
test_balanced_accuracy = np.empty(len(rs))
train_balanced_accuracy = np.empty(len(rs))



for j in rs:
    X_train, X_test, y_train, y_test = train_test_split(
        data_1, target_1, test_size=0.2, stratify=target_1, random_state=j)
    
    # Create xgboost
    xgbo = XGBClassifier()
    
    # Create repeated stratified k-fold cross-validator
    cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=5, random_state=42)
    
    # Perform hyperparameter tuning with GridSearchCV
    grid_search = GridSearchCV(estimator=xgbo, param_grid=grid, n_jobs=-1, cv=cv, scoring=balanced_accuracy_scorer, error_score=0)
    grid_result = grid_search.fit(X_train, y_train.ravel())
    
    print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

    # Get the best Logsistic regression model from GridSearchCV
    best_rf = grid_search.best_estimator_

    # Compute balanced accuracy on the training and test data
    train_predictions = best_rf.predict(X_train)
    test_predictions = best_rf.predict(X_test)
    
    train_ba = balanced_accuracy_score(y_train, train_predictions)
    test_ba = balanced_accuracy_score(y_test, test_predictions)
    
    train_balanced_accuracy[j-1] = train_ba
    test_balanced_accuracy[j-1] = test_ba

In [ ]:
# Plot balanced accuracy for different random states
plt.plot(rs, test_balanced_accuracy, label='Testing dataset Balanced Accuracy')
plt.plot(rs, train_balanced_accuracy, label='Training dataset Balanced Accuracy')

plt.legend(fontsize=12)
plt.title('XGB Classifier Balanced Accuracy', fontsize=14)
plt.xlabel('Random State', fontsize=14)
plt.ylabel('Balanced Accuracy', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.savefig('XGB-Tuning-NORM-WWTP.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the value of best balanced accuracy and the corresponding random state
best_test_balanced_accuracy = np.max(test_balanced_accuracy)
best_random_state = rs[np.argmax(test_balanced_accuracy)]
print("Best Testing Balanced Accuracy:", best_test_balanced_accuracy)
print("Corresponding Random State:", best_random_state)

In [ ]:
print(train_balanced_accuracy)
print(test_balanced_accuracy)

In [ ]:
#Now running optimized model over the entire dataset
model = XGBClassifier(gamma = 1, learning_rate = 0.8, max_depth = 10, n_estimators = 70, reg_alpha = 0.8, reg_lambda = 0.5)
# fit the model
model.fit(data_1, target_1.ravel())

# get importance
importance = model.feature_importances_
# summarize feature importance
for i,v in enumerate(importance):
    print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
plt.bar([x for x in range(len(importance))], importance)
plt.show()

In [ ]:
# Initialize variables to store feature importances across iterations
all_importances = []

# Number of iterations
num_iterations = 100

for iteration in range(num_iterations):
    # Create and fit the model
    model = XGBClassifier(gamma = 1, learning_rate = 0.8, max_depth = 10, n_estimators = 70, reg_alpha = 0.8, reg_lambda = 0.5)
    model.fit(data_1, target_1.ravel())

    # Get importance and store in the list
    importance = model.feature_importances_
    all_importances.append(importance)

# Convert the list to a NumPy array for easy calculations
all_importances = np.array(all_importances)

# Calculate average and standard deviation of feature importances
average_importance = np.mean(all_importances, axis=0)
std_importance = np.std(all_importances, axis=0)

# Summarize and print feature importance statistics
for i, (avg_score, std_score) in enumerate(zip(average_importance, std_importance)):
    print(f'Feature {i}: Average Score: {avg_score:.5f}, Standard Deviation: {std_score:.5f}')

# Plot the average feature importance
plt.bar([x for x in range(len(average_importance))], average_importance, yerr=std_importance, capsize=5)
plt.xlabel('Feature Index')
plt.ylabel('Average Feature Importance')
plt.title('Average Feature Importance with Standard Deviation')
plt.show()

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

feature_importance_pairs = [(avg, std, name) for avg, std, name in zip(average_importance, std_importance, feature_names)]

# Sort the list by average importance in descending order
feature_importance_pairs.sort(reverse=True, key=lambda x: x[0])

# Print the output for the top 10 features
print("Top 10 Features:")
for avg, std, name in feature_importance_pairs[:10]:
    print(f'Feature: {name}, Mean Score: {avg:.5f}, Standard Deviation: {std:.5f}')

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Get indices of the top 10 features based on mean importance
top_10_indices = np.argsort(average_importance)[::-1][:10]

# Print top 10 features along with their average and standard deviation
print("Top 10 Features:")
for i in top_10_indices:
    print(f'Feature: {feature_names[i]}, Mean Score: {average_importance[i]:.5f}, Standard Deviation: {std_importance[i]:.5f}')

In [ ]:
# Rank features by importance
sorted_feature_importance_pairs = sorted(feature_importance_pairs, reverse=True, key=lambda x: x[0])

# Extract feature names and importances
sorted_importances, _, sorted_feature_names = zip(*sorted_feature_importance_pairs)

# Convert feature importances to a numeric data type
sorted_importances = np.array(sorted_importances, dtype=float)

# Calculate the difference between subsequent feature importances
importance_diff = np.diff(sorted_importances)

# Find the index where the largest decrease occurs
largest_decrease_index = np.argmax(importance_diff)


# Store the top features and scores in a DataFrame
df_top_features = pd.DataFrame({'Feature': [], 'Score': []})

# Print the top features until the largest decrease
print("Top Features with Significant Importance:")
for i, (avg, std, name) in enumerate(sorted_feature_importance_pairs[:largest_decrease_index + 1], 1):
    try:
        avg_float = float(avg)
        print(f"{i}. Feature: {name}, Mean Score: {avg_float:.5f}, Standard Deviation: {std:.5f}")
        df_top_features = df_top_features.append({'Feature': name, 'Score': avg_float}, ignore_index=True)
    except ValueError:
        print(f"{i}. Feature: {name}, Mean Score: {avg}, Standard Deviation: {std} (non-numeric)")

# Save the df_top_features DataFrame to an Excel file
df_top_features.to_excel('top_features_scores_norm_WWTP.xlsx', index=False)

# Plot feature importance
plt.figure(figsize=(200, 60))
plt.bar(range(len(sorted_importances)), sorted_importances, tick_label=sorted_feature_names)
plt.xticks(rotation=90)
plt.title("Feature Importances")
# Save the figure as a PNG file
plt.savefig('feature_importances_plot_ORDERED_norm_WWTP.png', bbox_inches='tight')

# Show the plot (optional)
plt.show()

In [ ]:
class_names=np.array([0.0,1.0])

#Generating confusion matrices (for entire dataset; normalized BAs)
def evaluate_model(data_x, data_y):
    k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

    predicted_targets = np.array([])
    actual_targets = np.array([])

    for train_ix, test_ix in k_fold.split(data_x, data_y):
        train_x, train_y, test_x, test_y = data_x[train_ix], data_y[train_ix], data_x[test_ix], data_y[test_ix]

        # Fit the classifier
        model_cv = XGBClassifier(gamma = 1, learning_rate = 0.8, max_depth = 10, n_estimators = 70, reg_alpha = 0.8, reg_lambda = 0.5)
        classifier = model_cv.fit(train_x, train_y.ravel())

        # Predict the labels of the test set samples
        predicted_labels = classifier.predict(test_x)

        predicted_targets = np.append(predicted_targets, predicted_labels)
        actual_targets = np.append(actual_targets, test_y)

    return predicted_targets, actual_targets

In [ ]:
def plot_confusion_matrix(predicted_labels_list, y_test_list):
    cnf_matrix = confusion_matrix(y_test_list, predicted_labels_list)
    np.set_printoptions(precision=2)

    # Plot non-normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, title='Confusion matrix, without normalization')
    plt.show()

    # Plot normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, normalize=True, title='Normalized confusion matrix')
    plt.show()

In [ ]:
def generate_confusion_matrix(cnf_matrix, classes, normalize=False, title='Confusion matrix'):
    if normalize:
        cnf_matrix = cnf_matrix.astype('float') / cnf_matrix.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.imshow(cnf_matrix, interpolation='nearest', cmap=plt.get_cmap('Blues'))
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cnf_matrix.max() / 2.
    
    for i, j in itertools.product(range(cnf_matrix.shape[0]), range(cnf_matrix.shape[1])):
        plt.text(j, i, format(cnf_matrix[i, j], fmt), horizontalalignment="center",
                 color="white" if cnf_matrix[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

    return cnf_matrix

In [ ]:
predicted_target, actual_target = evaluate_model(data_1, target_1)
plot_confusion_matrix(predicted_target, actual_target)

### PP

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
#print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#Running randomized CV to narrow grid for GridSearch CV
#Modified from here: https://towardsdatascience.com/hyperparameter-tuning-the-random-forest-in-python-using-scikit-learn-28d2aa77dd74
random_grid = {'gamma': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4, 200],
              'learning_rate': [0.01, 0.03, 0.06, 0.1, 0.15, 0.2, 0.25, 0.300000012, 0.4, 0.5, 0.6, 0.7],
              'max_depth': [5,6,7,8,9,10,11,12,13,14],
              'n_estimators': [50,65,80,100,115,130,150],
              'reg_alpha': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200],
              'reg_lambda': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200]}

print(random_grid)

In [ ]:
# Use the random grid to search for best hyperparameters
# First create the base model to tune
model_rnd = XGBClassifier()
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = model_rnd, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
rf_random.fit(data_1, target_1)
print(rf_random.best_params_)

In [ ]:
# Split into test and training, and evaluate optimal hyperparameters for Random Forest
totalR = 10
rs = np.arange(1, totalR + 1)

#Define hyperparameters
gamma = [0, 0.4, 0.6]
learning_rate = [0.2, 0.5, 0.15]
max_depth = [5,10, 12]
n_estimators = [40, 50, 115]
reg_alpha = [0.4, 0.8, 10]
reg_lambda = [0.2, 0.5, 0.8]

grid = dict(gamma=gamma, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, reg_alpha=reg_alpha, reg_lambda=reg_lambda)

balanced_accuracy_scorer = make_scorer(balanced_accuracy_score)
# Initialize arrays to store results
test_balanced_accuracy = np.empty(len(rs))
train_balanced_accuracy = np.empty(len(rs))



for j in rs:
    X_train, X_test, y_train, y_test = train_test_split(
        data_1, target_1, test_size=0.2, stratify=target_1, random_state=j)
    
    # Create xgboost
    xgbo = XGBClassifier()
    
    # Create repeated stratified k-fold cross-validator
    cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=5, random_state=42)
    
    # Perform hyperparameter tuning with GridSearchCV
    grid_search = GridSearchCV(estimator=xgbo, param_grid=grid, n_jobs=-1, cv=cv, scoring=balanced_accuracy_scorer, error_score=0)
    grid_result = grid_search.fit(X_train, y_train.ravel())
    
    print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

    # Get the best Logsistic regression model from GridSearchCV
    best_rf = grid_search.best_estimator_

    # Compute balanced accuracy on the training and test data
    train_predictions = best_rf.predict(X_train)
    test_predictions = best_rf.predict(X_test)
    
    train_ba = balanced_accuracy_score(y_train, train_predictions)
    test_ba = balanced_accuracy_score(y_test, test_predictions)
    
    train_balanced_accuracy[j-1] = train_ba
    test_balanced_accuracy[j-1] = test_ba

In [ ]:
# Plot balanced accuracy for different random states
plt.plot(rs, test_balanced_accuracy, label='Testing dataset Balanced Accuracy')
plt.plot(rs, train_balanced_accuracy, label='Training dataset Balanced Accuracy')

plt.legend(fontsize=12)
plt.title('XGB Classifier Balanced Accuracy', fontsize=14)
plt.xlabel('Random State', fontsize=14)
plt.ylabel('Balanced Accuracy', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.savefig('XGB-Tuning-NORM-PP.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the value of best balanced accuracy and the corresponding random state
best_test_balanced_accuracy = np.max(test_balanced_accuracy)
best_random_state = rs[np.argmax(test_balanced_accuracy)]
print("Best Testing Balanced Accuracy:", best_test_balanced_accuracy)
print("Corresponding Random State:", best_random_state)

In [ ]:
print(train_balanced_accuracy)
print(test_balanced_accuracy)

In [ ]:
#Now running optimized model over the entire dataset
model = XGBClassifier(gamma = 0, learning_rate = 0.5, max_depth = 5, n_estimators = 40, reg_alpha = 0.4, reg_lambda = 0.5)
# fit the model
model.fit(data_1, target_1.ravel())

# get importance
importance = model.feature_importances_
# summarize feature importance
for i,v in enumerate(importance):
    print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
plt.bar([x for x in range(len(importance))], importance)
plt.show()

In [ ]:
# Initialize variables to store feature importances across iterations
all_importances = []

# Number of iterations
num_iterations = 100

for iteration in range(num_iterations):
    # Create and fit the model
    model = XGBClassifier(gamma = 0, learning_rate = 0.5, max_depth = 5, n_estimators = 40, reg_alpha = 0.4, reg_lambda = 0.5)
    model.fit(data_1, target_1.ravel())

    # Get importance and store in the list
    importance = model.feature_importances_
    all_importances.append(importance)

# Convert the list to a NumPy array for easy calculations
all_importances = np.array(all_importances)

# Calculate average and standard deviation of feature importances
average_importance = np.mean(all_importances, axis=0)
std_importance = np.std(all_importances, axis=0)

# Summarize and print feature importance statistics
for i, (avg_score, std_score) in enumerate(zip(average_importance, std_importance)):
    print(f'Feature {i}: Average Score: {avg_score:.5f}, Standard Deviation: {std_score:.5f}')

# Plot the average feature importance
plt.bar([x for x in range(len(average_importance))], average_importance, yerr=std_importance, capsize=5)
plt.xlabel('Feature Index')
plt.ylabel('Average Feature Importance')
plt.title('Average Feature Importance with Standard Deviation')
plt.show()

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

feature_importance_pairs = [(avg, std, name) for avg, std, name in zip(average_importance, std_importance, feature_names)]

# Sort the list by average importance in descending order
feature_importance_pairs.sort(reverse=True, key=lambda x: x[0])

# Print the output for the top 10 features
print("Top 10 Features:")
for avg, std, name in feature_importance_pairs[:10]:
    print(f'Feature: {name}, Mean Score: {avg:.5f}, Standard Deviation: {std:.5f}')

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Get indices of the top 10 features based on mean importance
top_10_indices = np.argsort(average_importance)[::-1][:10]

# Print top 10 features along with their average and standard deviation
print("Top 10 Features:")
for i in top_10_indices:
    print(f'Feature: {feature_names[i]}, Mean Score: {average_importance[i]:.5f}, Standard Deviation: {std_importance[i]:.5f}')

In [ ]:
# Rank features by importance
sorted_feature_importance_pairs = sorted(feature_importance_pairs, reverse=True, key=lambda x: x[0])

# Extract feature names and importances
sorted_importances, _, sorted_feature_names = zip(*sorted_feature_importance_pairs)

# Convert feature importances to a numeric data type
sorted_importances = np.array(sorted_importances, dtype=float)

# Calculate the difference between subsequent feature importances
importance_diff = np.diff(sorted_importances)

# Find the index where the largest decrease occurs
largest_decrease_index = np.argmax(importance_diff)


# Store the top features and scores in a DataFrame
df_top_features = pd.DataFrame({'Feature': [], 'Score': []})

# Print the top features until the largest decrease
print("Top Features with Significant Importance:")
for i, (avg, std, name) in enumerate(sorted_feature_importance_pairs[:largest_decrease_index + 1], 1):
    try:
        avg_float = float(avg)
        print(f"{i}. Feature: {name}, Mean Score: {avg_float:.5f}, Standard Deviation: {std:.5f}")
        df_top_features = df_top_features.append({'Feature': name, 'Score': avg_float}, ignore_index=True)
    except ValueError:
        print(f"{i}. Feature: {name}, Mean Score: {avg}, Standard Deviation: {std} (non-numeric)")

# Save the df_top_features DataFrame to an Excel file
df_top_features.to_excel('top_features_scores_norm_PP.xlsx', index=False)

# Plot feature importance
plt.figure(figsize=(200, 60))
plt.bar(range(len(sorted_importances)), sorted_importances, tick_label=sorted_feature_names)
plt.xticks(rotation=90)
plt.title("Feature Importances")
# Save the figure as a PNG file
plt.savefig('feature_importances_plot_ORDERED_norm_PP.png', bbox_inches='tight')

# Show the plot (optional)
plt.show()

In [ ]:
class_names=np.array([0.0,1.0])

#Generating confusion matrices (for entire dataset; normalized BAs)
def evaluate_model(data_x, data_y):
    k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

    predicted_targets = np.array([])
    actual_targets = np.array([])

    for train_ix, test_ix in k_fold.split(data_x, data_y):
        train_x, train_y, test_x, test_y = data_x[train_ix], data_y[train_ix], data_x[test_ix], data_y[test_ix]

        # Fit the classifier
        model_cv = XGBClassifier(gamma = 0, learning_rate = 0.5, max_depth = 5, n_estimators = 40, reg_alpha = 0.4, reg_lambda = 0.5)
        classifier = model_cv.fit(train_x, train_y.ravel())

        # Predict the labels of the test set samples
        predicted_labels = classifier.predict(test_x)

        predicted_targets = np.append(predicted_targets, predicted_labels)
        actual_targets = np.append(actual_targets, test_y)

    return predicted_targets, actual_targets

In [ ]:
def plot_confusion_matrix(predicted_labels_list, y_test_list):
    cnf_matrix = confusion_matrix(y_test_list, predicted_labels_list)
    np.set_printoptions(precision=2)

    # Plot non-normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, title='Confusion matrix, without normalization')
    plt.show()

    # Plot normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, normalize=True, title='Normalized confusion matrix')
    plt.show()

In [ ]:
def generate_confusion_matrix(cnf_matrix, classes, normalize=False, title='Confusion matrix'):
    if normalize:
        cnf_matrix = cnf_matrix.astype('float') / cnf_matrix.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.imshow(cnf_matrix, interpolation='nearest', cmap=plt.get_cmap('Blues'))
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cnf_matrix.max() / 2.
    
    for i, j in itertools.product(range(cnf_matrix.shape[0]), range(cnf_matrix.shape[1])):
        plt.text(j, i, format(cnf_matrix[i, j], fmt), horizontalalignment="center",
                 color="white" if cnf_matrix[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

    return cnf_matrix

In [ ]:
predicted_target, actual_target = evaluate_model(data_1, target_1)
plot_confusion_matrix(predicted_target, actual_target)

### PG

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
#print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#Running randomized CV to narrow grid for GridSearch CV
#Modified from here: https://towardsdatascience.com/hyperparameter-tuning-the-random-forest-in-python-using-scikit-learn-28d2aa77dd74
random_grid = {'gamma': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4, 200],
              'learning_rate': [0.01, 0.03, 0.06, 0.1, 0.15, 0.2, 0.25, 0.300000012, 0.4, 0.5, 0.6, 0.7],
              'max_depth': [5,6,7,8,9,10,11,12,13,14],
              'n_estimators': [50,65,80,100,115,130,150],
              'reg_alpha': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200],
              'reg_lambda': [0,0.1,0.2,0.4,0.8,1.6,3.2,6.4,12.8,25.6,51.2,102.4,200]}

print(random_grid)

In [ ]:
# Use the random grid to search for best hyperparameters
# First create the base model to tune
model_rnd = XGBClassifier()
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = model_rnd, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
rf_random.fit(data_1, target_1)
print(rf_random.best_params_)

In [ ]:
# Split into test and training, and evaluate optimal hyperparameters for Random Forest
totalR = 10
rs = np.arange(1, totalR + 1)

#Define hyperparameters
gamma = [0, 0.2, 0.4]
learning_rate = [0, 0.2, 0.4]
max_depth = [10, 11, 12]
n_estimators = [90, 100, 110]
reg_alpha = [0.2, 0.8, 1.0]
reg_lambda = [0.1, 0.2, 0.3]

grid = dict(gamma=gamma, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, reg_alpha=reg_alpha, reg_lambda=reg_lambda)

balanced_accuracy_scorer = make_scorer(balanced_accuracy_score)
# Initialize arrays to store results
test_balanced_accuracy = np.empty(len(rs))
train_balanced_accuracy = np.empty(len(rs))



for j in rs:
    X_train, X_test, y_train, y_test = train_test_split(
        data_1, target_1, test_size=0.2, stratify=target_1, random_state=j)
    
    # Create xgboost
    xgbo = XGBClassifier()
    
    # Create repeated stratified k-fold cross-validator
    cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=5, random_state=42)
    
    # Perform hyperparameter tuning with GridSearchCV
    grid_search = GridSearchCV(estimator=xgbo, param_grid=grid, n_jobs=-1, cv=cv, scoring=balanced_accuracy_scorer, error_score=0)
    grid_result = grid_search.fit(X_train, y_train.ravel())
    
    print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

    # Get the best Logsistic regression model from GridSearchCV
    best_rf = grid_search.best_estimator_

    # Compute balanced accuracy on the training and test data
    train_predictions = best_rf.predict(X_train)
    test_predictions = best_rf.predict(X_test)
    
    train_ba = balanced_accuracy_score(y_train, train_predictions)
    test_ba = balanced_accuracy_score(y_test, test_predictions)
    
    train_balanced_accuracy[j-1] = train_ba
    test_balanced_accuracy[j-1] = test_ba

In [ ]:
# Plot balanced accuracy for different random states
plt.plot(rs, test_balanced_accuracy, label='Testing dataset Balanced Accuracy')
plt.plot(rs, train_balanced_accuracy, label='Training dataset Balanced Accuracy')

plt.legend(fontsize=12)
plt.title('XGB Classifier Balanced Accuracy', fontsize=14)
plt.xlabel('Random State', fontsize=14)
plt.ylabel('Balanced Accuracy', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.savefig('XGB-Tuning-NORM-PG.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the value of best balanced accuracy and the corresponding random state
best_test_balanced_accuracy = np.max(test_balanced_accuracy)
best_random_state = rs[np.argmax(test_balanced_accuracy)]
print("Best Testing Balanced Accuracy:", best_test_balanced_accuracy)
print("Corresponding Random State:", best_random_state)

In [ ]:
print(train_balanced_accuracy)
print(test_balanced_accuracy)

In [ ]:
#Now running optimized model over the entire dataset
model = XGBClassifier(gamma = 0, learning_rate = 0.4, max_depth = 10, n_estimators = 90, reg_alpha = 0.2, reg_lambda = 0.2)
# fit the model
model.fit(data_1, target_1.ravel())

# get importance
importance = model.feature_importances_
# summarize feature importance
for i,v in enumerate(importance):
    print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
plt.bar([x for x in range(len(importance))], importance)
plt.show()

In [ ]:
# Initialize variables to store feature importances across iterations
all_importances = []

# Number of iterations
num_iterations = 100

for iteration in range(num_iterations):
    # Create and fit the model
    model = XGBClassifier(gamma = 0, learning_rate = 0.4, max_depth = 10, n_estimators = 90, reg_alpha = 0.2, reg_lambda = 0.2)
    model.fit(data_1, target_1.ravel())

    # Get importance and store in the list
    importance = model.feature_importances_
    all_importances.append(importance)

# Convert the list to a NumPy array for easy calculations
all_importances = np.array(all_importances)

# Calculate average and standard deviation of feature importances
average_importance = np.mean(all_importances, axis=0)
std_importance = np.std(all_importances, axis=0)

# Summarize and print feature importance statistics
for i, (avg_score, std_score) in enumerate(zip(average_importance, std_importance)):
    print(f'Feature {i}: Average Score: {avg_score:.5f}, Standard Deviation: {std_score:.5f}')

# Plot the average feature importance
plt.bar([x for x in range(len(average_importance))], average_importance, yerr=std_importance, capsize=5)
plt.xlabel('Feature Index')
plt.ylabel('Average Feature Importance')
plt.title('Average Feature Importance with Standard Deviation')
plt.show()

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

feature_importance_pairs = [(avg, std, name) for avg, std, name in zip(average_importance, std_importance, feature_names)]

# Sort the list by average importance in descending order
feature_importance_pairs.sort(reverse=True, key=lambda x: x[0])

# Print the output for the top 10 features
print("Top 10 Features:")
for avg, std, name in feature_importance_pairs[:10]:
    print(f'Feature: {name}, Mean Score: {avg:.5f}, Standard Deviation: {std:.5f}')

In [ ]:
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Get indices of the top 10 features based on mean importance
top_10_indices = np.argsort(average_importance)[::-1][:10]

# Print top 10 features along with their average and standard deviation
print("Top 10 Features:")
for i in top_10_indices:
    print(f'Feature: {feature_names[i]}, Mean Score: {average_importance[i]:.5f}, Standard Deviation: {std_importance[i]:.5f}')

In [ ]:
# Rank features by importance
sorted_feature_importance_pairs = sorted(feature_importance_pairs, reverse=True, key=lambda x: x[0])

# Extract feature names and importances
sorted_importances, _, sorted_feature_names = zip(*sorted_feature_importance_pairs)

# Convert feature importances to a numeric data type
sorted_importances = np.array(sorted_importances, dtype=float)

# Calculate the difference between subsequent feature importances
importance_diff = np.diff(sorted_importances)

# Find the index where the largest decrease occurs
largest_decrease_index = np.argmax(importance_diff)


# Store the top features and scores in a DataFrame
df_top_features = pd.DataFrame({'Feature': [], 'Score': []})

# Print the top features until the largest decrease
print("Top Features with Significant Importance:")
for i, (avg, std, name) in enumerate(sorted_feature_importance_pairs[:largest_decrease_index + 1], 1):
    try:
        avg_float = float(avg)
        print(f"{i}. Feature: {name}, Mean Score: {avg_float:.5f}, Standard Deviation: {std:.5f}")
        df_top_features = df_top_features.append({'Feature': name, 'Score': avg_float}, ignore_index=True)
    except ValueError:
        print(f"{i}. Feature: {name}, Mean Score: {avg}, Standard Deviation: {std} (non-numeric)")

# Save the df_top_features DataFrame to an Excel file
df_top_features.to_excel('top_features_scores_norm_PG.xlsx', index=False)

# Plot feature importance
plt.figure(figsize=(200, 60))
plt.bar(range(len(sorted_importances)), sorted_importances, tick_label=sorted_feature_names)
plt.xticks(rotation=90)
plt.title("Feature Importances")
# Save the figure as a PNG file
plt.savefig('feature_importances_plot_ORDERED_norm_PG.png', bbox_inches='tight')

# Show the plot (optional)
plt.show()

In [ ]:
class_names=np.array([0.0,1.0])

#Generating confusion matrices (for entire dataset; normalized BAs)
def evaluate_model(data_x, data_y):
    k_fold = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)

    predicted_targets = np.array([])
    actual_targets = np.array([])

    for train_ix, test_ix in k_fold.split(data_x, data_y):
        train_x, train_y, test_x, test_y = data_x[train_ix], data_y[train_ix], data_x[test_ix], data_y[test_ix]

        # Fit the classifier
        model_cv = XGBClassifier(gamma = 0.2, learning_rate = 0.4, max_depth = 10, n_estimators = 90, reg_alpha = 0.2, reg_lambda = 0.1)
        classifier = model_cv.fit(train_x, train_y.ravel())

        # Predict the labels of the test set samples
        predicted_labels = classifier.predict(test_x)

        predicted_targets = np.append(predicted_targets, predicted_labels)
        actual_targets = np.append(actual_targets, test_y)

    return predicted_targets, actual_targets

In [ ]:
def plot_confusion_matrix(predicted_labels_list, y_test_list):
    cnf_matrix = confusion_matrix(y_test_list, predicted_labels_list)
    np.set_printoptions(precision=2)

    # Plot non-normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, title='Confusion matrix, without normalization')
    plt.show()

    # Plot normalized confusion matrix
    plt.figure()
    generate_confusion_matrix(cnf_matrix, classes=class_names, normalize=True, title='Normalized confusion matrix')
    plt.show()

In [ ]:
def generate_confusion_matrix(cnf_matrix, classes, normalize=False, title='Confusion matrix'):
    if normalize:
        cnf_matrix = cnf_matrix.astype('float') / cnf_matrix.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.imshow(cnf_matrix, interpolation='nearest', cmap=plt.get_cmap('Blues'))
    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cnf_matrix.max() / 2.
    
    for i, j in itertools.product(range(cnf_matrix.shape[0]), range(cnf_matrix.shape[1])):
        plt.text(j, i, format(cnf_matrix[i, j], fmt), horizontalalignment="center",
                 color="white" if cnf_matrix[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

    return cnf_matrix

In [ ]:
predicted_target, actual_target = evaluate_model(data_1, target_1)
plot_confusion_matrix(predicted_target, actual_target)